In [18]:
from tqdm import tqdm
from architecture import Embeddings, PositionalEncoding, Encoder, Batch,BinaryClassifier, PGKPR
import torch.nn as nn
import torch
import math
from utils import loadVocab, vocab_path

In [19]:
mini_input_size = 64

ecbplus_inputs = torch.load("ecbplus_tokenized_inputs.pt")
ecbplus_mini_inputs = torch.clone(ecbplus_inputs[:mini_input_size])

ecbplus_labels = torch.load("ecbplus_labels.pt")
ecbplus_mini_labels = torch.clone(ecbplus_labels[:mini_input_size])

In [30]:
ecbplus_mini_inputs.shape

torch.Size([64, 200])

In [31]:
LOWERCASE = True
token2idx, idx2token = loadVocab(vocab_path, lowercase=LOWERCASE) # I should go back to the first stage and understand better how the vocab is built. However, it is not subtokened
vocab_size = len(token2idx)


In [32]:
# Encode data
#Initialize
encoder = Encoder(num_layers=6, d_model=512, nhead=8, d_ff=2048, dropout=0.1, return_layer=6)

#Manual run 
embed_test = Embeddings(512, vocab_size)
pos_encod_test = PositionalEncoding(512, 0.1, double=False)

src_embed_pos = nn.Sequential(Embeddings(512, vocab_size), #Token id to embedding
                                           PositionalEncoding(512, 0.1, double=False))

ecbplus_mini_inputs_mask = (ecbplus_mini_inputs != 0).unsqueeze(-2)

xx = src_embed_pos.forward(ecbplus_mini_inputs)
x = embed_test.forward(ecbplus_mini_inputs)
y  = pos_encod_test.forward(x)

final_encoder_layer, intermediate_encoder_layer = encoder.forward(y,ecbplus_mini_inputs_mask)

## 'Decode' Part for CDCR.

binary_classifier = BinaryClassifier(hidden_size=512, layer_norm_eps=1e-12)
output = binary_classifier(final_encoder_layer)



In [36]:
# model
#D_model is hidden_state size
D_MODEL = 64 #Change temporariliy to a small size from 512 to run in small laptop

MID_LOSS = True
DOUBLE_POSITION = False
tqdm_print_step = 1
model_save_path = 'models/'
BATCH_SIZE = 8
VAL_BATCH_SIZE = 8 

In [41]:
# Load model and link optimizer
model = PGKPR(d_model=D_MODEL, vocab=vocab_size, mid_loss=MID_LOSS, double_pe=DOUBLE_POSITION,
                          shared_emb=True)


model_opt = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)

In [ ]:
# I will continue to analyze whether I need that Batch masking which seems to be more for next sentence prediction
# It is indeed like that but I like the concept of the batch doing the marking. I will replicate it 

In [43]:
train_data = ecbplus_mini_inputs, ecbplus_mini_labels 

val_data = ecbplus_mini_inputs, ecbplus_mini_labels #Temporarily same data to validate run

In [44]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

train_data = TensorDataset(*train_data)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=BATCH_SIZE)

val_data = TensorDataset(*val_data)
val_dataloader = DataLoader(val_data, batch_size=VAL_BATCH_SIZE) 

dataloader = {'train': train_dataloader, 'val': val_dataloader}

In [48]:
def run_once(model, batch,loss_function,optimizer):
    predicted_probs= model.forward(src=batch.src,src_mask=batch.src_mask)
    predicted_probs = predicted_probs.squeeze()
    optimizer.zero_grad()


    loss = loss_function(predicted_probs, batch.trg)
    loss.backward()
    optimizer.step()
    return loss

loss_function = torch.nn.BCELoss()
best_loss = float("inf")

for epoch in range(5): #TODO define epochs parameter from config file
    # scheduler.step()
    print(f'\n EPOCH {epoch}')
    model.train()
    total_loss = 0
    #cdcr_loss, pi_loss= 0, 0
    
    for phase in ['train', 'val']:
        if phase=='train':
            model.train()
        else:
            model.eval()

        with tqdm(total=len(dataloader[phase])) as pbar: 
            #Dataloader should be controlled to send a batch of one task only.
            for i, data in enumerate(dataloader[phase]):
                src, tgt= data #dataloader must have source and target, in my case the label
                batch = Batch(src, tgt)
                # src, tgt= src.cuda() tgt.cuda()#TODO send everything to device. I could adapt it to finding device first
                binary_loss = run_once(model,batch,loss_function,model_opt)
                total_loss += binary_loss #It would be a good idea to monitor both losses separately
                
                if i % tqdm_print_step == 0:
                        pbar.update(tqdm_print_step)
                        pbar.set_description(
                            f'train loss: {binary_loss:.4f}')
                        
        epoch_loss = total_loss / len(dataloader[phase]) #Mean relative to dataloader size

        print(f"epoch loss: {epoch_loss}")

        if phase == 'val' and (epoch_loss < best_loss or epoch % 10 == 0) and epoch > 5:
            print("save model")
            best_loss = epoch_loss
            torch.save(model.state_dict(), f'{model_save_path}/epoch{epoch}_loss_{epoch_loss}.pkl')



 EPOCH 0


  0%|          | 0/8 [00:00<?, ?it/s]

train loss: 0.5056: 100%|██████████| 8/8 [00:08<00:00,  1.07s/it]


epoch loss: 0.556523323059082


train loss: 0.0660: 100%|██████████| 8/8 [00:06<00:00,  1.29it/s]


...

epoch loss: 0.7120340466499329

 EPOCH 4


train loss: 0.1009: 100%|██████████| 8/8 [00:07<00:00,  1.04it/s]


epoch loss: 0.34834134578704834


train loss: 0.1280: 100%|██████████| 8/8 [00:06<00:00,  1.27it/s]

epoch loss: 0.7121716737747192


In [ ]:
# Types seem to be fine, now I have to add the the optimizer step.